In [5]:
import pandas as pd
import numpy as np

data = pd.read_csv('../data/processed/punjab_district_weekly_merged.csv')

def encyclicEncoding(data):

    data["month_sin"] = np.sin(2 * np.pi * data["month_num"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month_num"] / 12)
    data["week_sin"] = np.sin(2 * np.pi * data["epi_week"] / 52)
    data["week_cos"] = np.cos(2 * np.pi * data["epi_week"] / 52)

    return data

data=encyclicEncoding(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)



In [6]:
import pandas as pd
import numpy as np

data = pd.read_csv('../data/processed/updated-Punjab.csv')


def HandleDataInconsistency(data):
        
    data["district"] = data["district"].ffill()
    data["month"] = data["month"].ffill()

    data["cases"] = data["cases"].interpolate(method="linear")
    data["deaths"] = data["deaths"].interpolate(method="linear")


    data = data.drop_duplicates()

    data["district"] = data["district"].str.lower().str.strip()


    data["cases"] = data["cases"].clip(lower=0)
    data["deaths"] = data["deaths"].clip(lower=0)


    data = data[(data["epi_week"] >= 1) & (data["epi_week"] <= 52)]
    data = data[(data["month_num"] >= 1) & (data["month_num"] <= 12)]


    data = data.sort_values(["district", "year", "epi_week"])


    assert data["month_num"].between(1, 12).all()
    assert data["epi_week"].between(1, 52).all()

    return data

data=HandleDataInconsistency(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)

In [2]:
import numpy as np
import pandas as pd

def lag_features(df):

    df=df.sort_values(by=["district", "epi_week","year"]).reset_index(drop=True)

    df["t1_cases"]=df.groupby("district")["cases"].shift(1)
    df["t2_cases"]=df.groupby("district")["cases"].shift(2)
    
    df["T2m_lag1"]=df.groupby("district")["T2M_mean"].shift(1)

    df["PRECTOTCORR_lag1"]=df.groupby("district")["PRECTOTCORR_sum"].shift(1)
    df["PRECTOTCORR_lag2"]=df.groupby("district")["PRECTOTCORR_sum"].shift(2)

    df=df.dropna(subset=["t1_cases", "t2_cases"]).reset_index(drop=True)
    return df

data = pd.read_csv('../data/processed/updated-Punjab.csv')
data=lag_features(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)

In [9]:
def water_proxy(df):

    df=df.sort_values(by=["district", "epi_week","year"]).reset_index(drop=True)

    df["water_proxy"]=(df.groupby("district")["PRECTOTCORR_sum"]
                        .transform(lambda elem:elem.rolling(window=2,min_periods=1).sum()))
    
    return df

data = pd.read_csv('../data/processed/updated-Punjab.csv')
data=water_proxy(data)
data.to_csv("../data/processed/updated-Punjab.csv", index=False)

In [3]:
#Chronological Data Split 70/15/15 for train, val, test
#here we cannot use sklearn.preprocessing's train_test_split because we need to maintain the temporal order of the data
import pandas as pd

df=pd.read_csv('../data/processed/updated-Punjab.csv')
# Sort by year first, then epi_week within each year 
df = df.sort_values(["year", "epi_week", "district"]).reset_index(drop=True) 
# Calculate split indices (70% / 15% / 15%) 
n = len(df) 
train_end = int(n * 0.70)   # First 70% for training 
val_end   = int(n * 0.85)   # Next 15% for validation 
# Index-based slicing — no shuffling, preserves time order 
train = df.iloc[:train_end].copy() 
val   = df.iloc[train_end:val_end].copy() 
test  = df.iloc[val_end:].copy() 
print(f"Total rows : {n}") 
print(f"Train rows : {len(train)} ({len(train)/n*100:.1f}%)") 
print(f"Val rows   : {len(val)}   ({len(val)/n*100:.1f}%)") 
print(f"Test rows  : {len(test)}  ({len(test)/n*100:.1f}%)")
    

Total rows : 8496
Train rows : 5947 (70.0%)
Val rows   : 1274   (15.0%)
Test rows  : 1275  (15.0%)


In [ ]:
#Feature Engineering 
#in our case,the features are mostly the derived and engineered rows like the lag features,water proxy,encycling-features and the weather condition while our target is the no. of cases

Features = [ 
"T2M_mean",          # Current temperature  
"RH2M_mean",          # Current humidity 
"PRECTOTCORR_sum",    # Current precipitation 
"WS10M_mean",        # Current wind speed 
"T2M_lag1",          # Previous week temperature (delayed effect) 
"PRECTOTCORR_lag1",   # Previous week rainfall 
"PRECTOTCORR_lag2",   # Two-week rainfall (standing water) 
"t1_cases",           # Cases last week (autoregressive) 
"t2_cases",           # Cases two weeks ago 
"water_proxy",        
"month_sin",          
"month_cos",          
"week_sin",          
"week_cos"            
]

Target = "cases"  # Number of dengue cases to predict
